# Build and Scale a Research Agent with LangChain Deep Agents and Amazon Bedrock AgentCore

This notebook walks you through building a competitive research agent that:

1. **Launches 3 browser-based researchers in parallel** — each in its own isolated AgentCore Browser MicroVM
2. **Analyzes findings with an interpreter** — generates comparison charts in an AgentCore Interpreter sandbox
3. **Builds institutional knowledge** — saves and recalls insights across sessions with AgentCore Memory
4. **Traces everything** — full nested observability with LangSmith

**How it works:** The coordinator agent receives your query, checks AgentCore Memory
for past research, then launches three research subagents in parallel via `task()`.
Each researcher browses a competitor's website in its own isolated MicroVM. Once
the three return findings, an analyst subagent generates a comparison chart in
a sandboxed interpreter. Key insights are saved to memory for future sessions.

> **Optional:** Part 2 at the bottom of this notebook shows how to deploy this same agent
> to **Amazon Bedrock AgentCore Runtime** using the AgentCore CLI, so it runs as a
> managed, session-isolated service instead of locally in the notebook.

## Prerequisites

- AWS account with Amazon Bedrock AgentCore access enabled
- AWS credentials exported as environment variables:
  ```bash
  export AWS_ACCESS_KEY_ID="..."
  export AWS_SECRET_ACCESS_KEY="..."
  export AWS_SESSION_TOKEN="..."  # if using temporary credentials
  export AWS_REGION="us-west-2"
- Python 3.11+
- (Optional) LangSmith account for observability
- (Optional) AgentCore Memory resource for cross-session knowledge

In [ ]:
# Install dependencies
%pip install -q "langchain-aws[tools]" deepagents bedrock-agentcore langgraph-checkpoint-aws

## Configuration

Set your AWS region, optional Memory ID, and optional LangSmith keys.
If you don't have an AgentCore Memory resource, leave `AGENTCORE_MEMORY_ID` empty.

In [ ]:
import os

# Required
os.environ["AWS_REGION"] = "us-west-2"

# Optional: AgentCore Memory (leave empty to skip)
os.environ["AGENTCORE_MEMORY_ID"] = ""  # e.g., "mem-abc123"

# Optional: LangSmith tracing (uncomment and fill in to enable)
#os.environ["LANGCHAIN_TRACING_V2"] = "true"
#os.environ["LANGCHAIN_API_KEY"] = ""  # your LangSmith API key
#os.environ["LANGCHAIN_PROJECT"] = "competitive-research-agent"

AWS_REGION = os.environ["AWS_REGION"]
MEMORY_ID = os.environ.get("AGENTCORE_MEMORY_ID", "")
ACTOR_ID = "competitive-research-agent" 

In [ ]:
import asyncio
from langchain_aws import ChatBedrockConverse
from langchain_aws.tools import create_browser_toolkit, create_code_interpreter_toolkit
from langchain_core.tools import tool
from deepagents import create_deep_agent
from langgraph.store.memory import InMemoryStore
from botocore.config import Config as BotoConfig

## Set up the model

You can use Claude through Amazon Bedrock, or swap for other LangChain-supported providers
(Anthropic API, Google Gemini, OpenAI) by changing a single line.

In [ ]:
# Accessing Claude Sonnet through Amazon Bedrock
model = ChatBedrockConverse(
    model="us.anthropic.claude-sonnet-4-6",
    region_name=AWS_REGION,
    config=BotoConfig(read_timeout=300),
)

# Alternative providers (uncomment one):
# from langchain_anthropic import ChatAnthropic
# model = ChatAnthropic(model="claude-sonnet-4-5-20250929")

# from langchain_google_genai import ChatGoogleGenerativeAI
# model = ChatGoogleGenerativeAI(model="gemini-2.5-pro")

print(f"Model ready: {model.model_id}")

## Define agent prompts

The coordinator manages the workflow: recall past research, delegate parallel research,
analyze, save insights. Each subagent has a focused prompt matching its role.

In [ ]:
COORDINATOR_PROMPT = """\
You are a competitive research coordinator. Your job is to orchestrate \
research across multiple competitors and produce a final comparison report.

## Your workflow:
1. FIRST, use recall_past_research to check if we have existing knowledge \
   about these competitors from previous sessions.
2. Parse the user's request to identify the competitors to research.
3. Launch one research subagent PER competitor, IN PARALLEL, using \
   the task() tool.
4. Once research subagents return, pass findings to the \
   data-analyst subagent to generate a comparison chart and report.
5. Use save_research_insights to persist key findings to AgentCore Memory.
6. Present the final report to the user.

## Important:
- Launch research tasks in parallel (multiple task() calls in one response).
- Do NOT use the take_screenshot tool.
"""

RESEARCHER_PROMPT = """\
You are a web research specialist. You research a single company by \
browsing their website and extracting structured information.

## Your process:
1. Navigate to the company's website using the navigate_browser tool.
2. Find pricing, features, and product information.
3. Extract text from relevant pages using the extract_text tool.
4. If a pricing page exists, navigate there and extract detailed tier information.

## Important:
- Do NOT use the take_screenshot tool. Use extract_text instead.

## Output format:
Return structured findings with: Company name, Pricing tiers, Notable features, Limitations.
"""

ANALYST_PROMPT = """\
You are a data analyst. You receive structured competitive research data \
and produce comparison charts and reports.

## Your process:
1. Parse the structured data provided.
2. Use execute_code to create a pandas DataFrame and matplotlib chart.
3. Save the chart as 'comparison_chart.png'.
4. Write a concise markdown comparison report.

Pre-installed: pandas, matplotlib, numpy.
"""

print("Prompts defined")

## Create browser toolkits

Each competitor gets its own `BrowserToolkit`, which means its own AgentCore Browser MicroVM.
You get complete isolation between parallel researchers.

The session manager's built-in `session_wait_timeout` handles concurrency when the LLM emits
multiple browser tool calls in a single turn.

In [ ]:
COMPETITORS = [
    ("GitHub", "https://github.com/pricing"),
    ("GitLab", "https://about.gitlab.com/pricing"),
    ("Bitbucket", "https://www.atlassian.com/software/bitbucket/pricing"),
]

toolkits_to_cleanup = []
research_subagents = []

for company_name, company_url in COMPETITORS:
    browser_toolkit, browser_tools = create_browser_toolkit(region=AWS_REGION)
    browser_toolkit.session_manager.session_wait_timeout = 60.0
    toolkits_to_cleanup.append(browser_toolkit)

    research_subagents.append({
        "name": f"research-{company_name.lower().replace(' ', '-')}",
        "description": f"Researches {company_name} by browsing {company_url}.",
        "system_prompt": RESEARCHER_PROMPT,
        "tools": browser_tools,
    })

print(f"Created {len(research_subagents)} research subagents, each with its own Browser MicroVM")

## Create the interpreter toolkit

The analyst subagent gets an AgentCore Interpreter, an isolated MicroVM with Python,
pandas, matplotlib, and numpy pre-installed.

In [ ]:
ci_toolkit, ci_tools = await create_code_interpreter_toolkit(region=AWS_REGION)
toolkits_to_cleanup.append(ci_toolkit)

analyst_subagent = {
    "name": "data-analyst",
    "description": "Analyzes competitor data, generates charts and reports.",
    "system_prompt": ANALYST_PROMPT,
    "tools": ci_tools,
}

print(f"Interpreter ready with {len(ci_tools)} tools: {[t.name for t in ci_tools]}")

## Create memory tools (optional)

If you have an AgentCore Memory resource, the coordinator gets three tools:
- **save_research_insights** — saves findings via `MemoryClient.create_event()`
- **recall_past_research** — semantic search via `MemoryClient.retrieve_memories()`
- **list_stored_memories** — lists stored memory records

If `AGENTCORE_MEMORY_ID` is empty, the agent runs without memory features.

> **Note:** Your AgentCore Memory resource must have at least one extraction strategy
> configured (such as `semanticMemoryStrategy`) for long-term recall to work.

In [ ]:
memory_tools = []

if MEMORY_ID:
    from bedrock_agentcore.memory import MemoryClient

    memory_client = MemoryClient(region_name=AWS_REGION)

    @tool
    async def save_research_insights(insights: str, session_id: str = "default") -> str:
        """Save competitive research insights to AgentCore long-term memory."""
        try:
            memory_client.create_event(
                memory_id=MEMORY_ID,
                actor_id=ACTOR_ID,
                session_id=session_id,
                messages=[
                    (f"Save these research insights:\n\n{insights}", "USER"),
                    ("Insights saved to long-term memory.", "ASSISTANT"),
                ],
            )
            return "Insights saved and are extracted into long-term memory."
        except Exception as e:
            return f"Failed to save: {e}"

    @tool
    async def recall_past_research(query: str, namespace_prefix: str = "/") -> str:
        """Search AgentCore long-term memory for past research insights."""
        try:
            memories = memory_client.retrieve_memories(
                memory_id=MEMORY_ID,
                namespace=namespace_prefix,
                query=query,
                top_k=5,
            )
            if not memories:
                return "No past research found matching that query."
            results = [f"**Result {i}:**\n{mem}" for i, mem in enumerate(memories, 1)]
            return f"Found {len(memories)} past memories:\n\n" + "\n\n---\n\n".join(results)
        except Exception as e:
            return f"Failed to retrieve memories: {e}"

    @tool
    async def list_stored_memories(namespace_prefix: str = "/", max_results: int = 10) -> str:
        """List stored long-term memory records."""
        try:
            response = memory_client.list_memory_records(
                memoryId=MEMORY_ID, namespace=namespace_prefix, maxResults=max_results,
            )
            records = response.get("memoryRecordSummaries", [])
            if not records:
                return "No memory records found."
            results = [
                f"- **{r.get('memoryRecordId', '?')}**: {r.get('memoryRecordText', 'N/A')[:200]}"
                for r in records
            ]
            return f"Found {len(records)} records:\n\n" + "\n".join(results)
        except Exception as e:
            return f"Failed: {e}"

    memory_tools = [save_research_insights, recall_past_research, list_stored_memories]
    print(f"Memory tools ready: {[t.name for t in memory_tools]}")
else:
    print("No AGENTCORE_MEMORY_ID set. Running without memory features.")

## Create the Deep Agent

The coordinator gets memory tools. Research subagents get browser tools.
The analyst gets interpreter tools. Each subagent type has access to only its own tools.

In [ ]:
agent = create_deep_agent(
    model=model,
    subagents=[*research_subagents, analyst_subagent],
    tools=memory_tools,
    system_prompt=COORDINATOR_PROMPT,
    name="competitive-research-coordinator",
    checkpointer=None,
    store=InMemoryStore(),
)

print("Agent created:")
print(f"  {len(research_subagents)} research subagents (browser tools)")
print(f"  1 analyst subagent (interpreter tools)")
print(f"  {len(memory_tools)} coordinator memory tools")

## Run the agent

The coordinator will:
1. Check memory for past research (if configured)
2. Launch 3 research subagents in parallel, each browsing a different competitor's pricing page
3. Pass findings to the analyst for chart generation
4. Save insights to memory (if configured)
5. Return the final report

**Expected runtime:** 4-6 minutes with Claude Sonnet.

In [ ]:
import uuid
import time

thread_id = f"notebook-{uuid.uuid4().hex[:8]}"
config = {
    "configurable": {
        "thread_id": thread_id,
        "actor_id": ACTOR_ID,
    }
}

start_time = time.time()
final_content = ""
tool_count = 0
shown_sessions = set()

print("⏳ Agent started...\n")

async for event in agent.astream_events(
    {"messages": [{"role": "user", "content": "Compare pricing for GitHub, GitLab, and Bitbucket"}]},
    config=config,
    version="v2",
):
    kind = event.get("event", "")
    name = event.get("name", "")
    data = event.get("data", {})

    if kind == "on_tool_start":
        elapsed = time.time() - start_time
        tool_input = data.get("input", {})

        if name == "task":
            subagent = tool_input.get("subagent_type", "unknown")
            print(f"  [{elapsed:5.1f}s] 🚀 Launching subagent: {subagent}")
        elif name == "navigate_browser":
            url = tool_input.get("url", "?")
            print(f"  [{elapsed:5.1f}s] 🌐 Browsing: {url}")
        elif name == "extract_text":
            print(f"  [{elapsed:5.1f}s] 📄 Extracting page text...")
        elif name == "execute_code":
            print(f"  [{elapsed:5.1f}s] 🐍 Running Python code...")
        elif name == "save_research_insights":
            print(f"  [{elapsed:5.1f}s] 💾 Saving to AgentCore Memory...")
        elif name == "recall_past_research":
            print(f"  [{elapsed:5.1f}s] 🧠 Searching AgentCore Memory...")
        else:
            print(f"  [{elapsed:5.1f}s] 🔧 {name}")
        tool_count += 1

    elif kind == "on_tool_end":
        elapsed = time.time() - start_time
        for tk in toolkits_to_cleanup:
            if not hasattr(tk, "session_manager"):
                continue
            for key, (client, browser, in_use) in tk.session_manager._async_sessions.items():
                if client.session_id and client.session_id not in shown_sessions:
                    shown_sessions.add(client.session_id)
                    print(f"           🔒 MicroVM session: {client.session_id}")
        if name == "task":
            output = str(data.get("output", ""))[:80]
            print(f"  [{elapsed:5.1f}s] ✅ Subagent done → {output}...")

    elif kind == "on_chat_model_stream":
        chunk = data.get("chunk")
        if chunk and hasattr(chunk, "content") and chunk.content:
            c = chunk.content
            if isinstance(c, list):
                for block in c:
                    if isinstance(block, dict) and block.get("type") == "text":
                        final_content += block["text"]
                    elif isinstance(block, str):
                        final_content += block
            elif isinstance(c, str):
                final_content += c

content = final_content if final_content else "No response captured. Check LangSmith trace."
elapsed = time.time() - start_time
print(f"\n✅ Done in {elapsed:.0f}s ({tool_count} tool calls)")
print(f"🔒 {len(shown_sessions)} isolated MicroVM sessions used")

## View the results

The agent's response contains the full competitive analysis rendered as markdown.
If LangSmith is configured, you can view the full nested trace showing parallel
subagent timing, tool calls, and token usage.

In [ ]:
from IPython.display import Markdown, display

display(Markdown(content))

# LangSmith trace link
if os.environ.get("LANGCHAIN_API_KEY"):
    project = os.environ.get("LANGCHAIN_PROJECT", "competitive-research-agent")
    print(f"\n View trace: https://smith.langchain.com/projects/p/{project}")

## Clean up

Stop browser and interpreter sessions to avoid incurring charges.
Sessions also auto-expire (1 hour for browsers, 15 minutes for interpreters).

In [ ]:
for tk in toolkits_to_cleanup:
    try:
        await tk.cleanup()
    except Exception as e:
        print(f"Cleanup warning: {e}")

print("AgentCore sessions cleaned up")

## Next steps

- **View your LangSmith trace** to see the full orchestration hierarchy, parallel timing, and token usage
- **Run again** to test memory recall. Wait 60+ seconds between runs for async memory extraction to complete.
- **Swap the model** by changing the `ChatBedrockConverse` line to `ChatAnthropic`, `ChatOpenAI`, or `ChatGoogleGenerativeAI`
- **Add more competitors** by adding entries to the `COMPETITORS` list
- **Adapt the pattern** by modifying prompts for due diligence, content research, or data pipeline orchestration
- **Deploy to AgentCore Runtime** — see Part 2 below to host this agent as a managed, session-isolated service

### Resources

- [LangChain Deep Agents documentation](https://docs.langchain.com/oss/python/deepagents/overview)
- [Amazon Bedrock AgentCore documentation](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/what-is-bedrock-agentcore.html)
- [Amazon Bedrock AgentCore pricing](https://aws.amazon.com/bedrock/pricing/)
- [LangSmith documentation](https://docs.langchain.com/langsmith/home)
- [langchain-aws GitHub repository](https://github.com/langchain-ai/langchain-aws)

---

# Part 2 (Optional): Deploy to Amazon Bedrock AgentCore Runtime

Up to this point, the agent has been running in-process inside this notebook. That's
great for development and iteration, but for production you typically want:

- **Managed hosting** — no notebook kernel to keep alive
- **Per-user session isolation** — each caller gets their own microVM
- **A stable endpoint** — invoke over HTTP from anywhere, with IAM auth
- **Built-in observability** — traces and logs flow to CloudWatch automatically

**Amazon Bedrock AgentCore Runtime** provides all of this. Your Deep Agents orchestrator,
the three parallel browser subagents, and the analyst all run **unchanged** — Runtime
just hosts them inside a container.

In this section we'll:
1. Verify prerequisites and install the AgentCore CLI
2. Scaffold a project and configure the deployment target
3. Replace the default agent code with our Deep Agents entrypoint
4. Update dependencies and generate the lock file
5. Deploy to AWS
6. Grant Browser/Interpreter permissions to the execution role
7. Invoke the deployed agent
8. View logs and traces
9. Tear down when finished

**Expected runtime for this section:** ~15-20 minutes total (CodeBuild container
build + first invocation cold start).

## Part 2 Prerequisites

In addition to the prerequisites at the top of the notebook, Part 2 requires:

### Tools
- **Node.js 20+** — the AgentCore CLI is an npm package. Verify: `node --version`
- **AWS CDK v2** — install globally: `npm install -g aws-cdk`
- **uv** — Python package manager for lock file generation. Verify: `uv --version`

### AWS Setup (one-time)
- **CDK bootstrapped** in your target region:
  ```bash
  cdk bootstrap aws://<account-id>/us-west-2
  ```
- **Model access** — Claude Sonnet 4 must be enabled in the Amazon Bedrock console.

### IAM Permissions

Your IAM identity (the credentials in your environment) needs permissions to:
- Assume CDK bootstrap roles (`sts:AssumeRole` on `cdk-*-deploy-role-*`, etc.)
- Invoke the deployed agent (`bedrock-agentcore:InvokeAgentRuntime`)
- Read logs and traces (`logs:StartLiveTail`, `logs:StartQuery`, etc.)
- Manage IAM inline policies (`iam:PutRolePolicy`) — needed for Step 6

The **Runtime execution role** (created automatically by CDK) needs additional
permissions for Browser and Code Interpreter APIs. The CDK template only grants
basic Bedrock model invocation and CloudWatch logging. We add the missing
permissions in Step 6 after deploy.

> **Charges:** Deploying creates a CloudFormation stack, ECR repository, IAM role,
> and AgentCore Runtime. Step 9 tears everything down. You pay for model tokens +
> Browser/Interpreter MicroVM time during invocations.

## Verify prerequisites

In [ ]:
import subprocess

checks = {
    "node": "node --version",
    "cdk": "cdk --version",
    "uv": "uv --version",
    "aws credentials": "aws sts get-caller-identity --query Account --output text",
}

all_ok = True
for name, cmd in checks.items():
    try:
        result = subprocess.run(cmd.split(), capture_output=True, text=True, timeout=10)
        if result.returncode != 0:
            raise RuntimeError(result.stderr.strip()[:100])
        version = result.stdout.strip().split("\n")[0][:60]
        print(f"  ✅ {name}: {version}")
    except Exception as e:
        print(f"  ❌ {name}: {e}")
        all_ok = False

if all_ok:
    print("\nAll prerequisites met. Ready for Part 2.")
else:
    print("\n⚠️  Fix the items above before continuing.")

## Step 1: Install the AgentCore CLI and scaffold the project

In [ ]:
# Install the AgentCore CLI globally (one-time)
!npm install -g @aws/agentcore
!agentcore --version

In [ ]:
# Scaffold the project:
#   --framework LangChain_LangGraph  → closest template to Deep Agents
#   --build Container                → required for browser/interpreter tooling
#   --memory none                    → we manage AgentCore Memory manually
#   --skip-install                   → we'll install deps + generate lock file ourselves

PROJECT_NAME = "DeepResearchAgent"

!agentcore create \
    --name {PROJECT_NAME} \
    --framework LangChain_LangGraph \
    --model-provider Bedrock \
    --memory none \
    --build Container \
    --defaults \
    --skip-install

# Inspect what was created
!ls -la {PROJECT_NAME}/
!ls -la {PROJECT_NAME}/app/{PROJECT_NAME}/

## Step 2: Configure the deployment target and install CDK dependencies

The scaffolded `aws-targets.json` needs your AWS account ID and region.
Since we used `--skip-install`, the CDK node modules also need to be installed.

In [ ]:
import json
import boto3

# Auto-detect account and region
account = boto3.client("sts").get_caller_identity()["Account"]
region = os.environ.get("AWS_REGION", "us-west-2")

# Write aws-targets.json
targets = [{"name": "default", "description": "Default target", "account": account, "region": region}]
targets_path = f"{PROJECT_NAME}/agentcore/aws-targets.json"
with open(targets_path, "w") as f:
    json.dump(targets, f, indent=2)
print(f"✅ aws-targets.json: account={account}, region={region}")

# Install CDK dependencies
!cd {PROJECT_NAME}/agentcore/cdk && npm install
print("\n✅ CDK dependencies installed")

## Step 3: Replace the default `main.py` with the Deep Agents entrypoint

The generated `main.py` uses a simple `create_react_agent`. We replace it with our
Deep Agents coordinator + 3 browser subagents + analyst setup from Part 1.

Key adjustments from Part 1:
- **Toolkits are created inside `invoke()`** so each session gets fresh isolated MicroVMs
- **Cleanup happens in `finally`** after each invocation

In [ ]:
main_py = '''\
"""Deep Agents competitive research agent on AgentCore Runtime."""
import os
from bedrock_agentcore.runtime import BedrockAgentCoreApp
from botocore.config import Config as BotoConfig
from langchain_aws import ChatBedrockConverse
from langchain_aws.tools import create_browser_toolkit, create_code_interpreter_toolkit
from deepagents import create_deep_agent
from langgraph.store.memory import InMemoryStore

AWS_REGION = os.environ.get("AWS_REGION", "us-west-2")

app = BedrockAgentCoreApp()
log = app.logger

COMPETITORS = [
    ("GitHub", "https://github.com/pricing"),
    ("GitLab", "https://about.gitlab.com/pricing"),
    ("Bitbucket", "https://www.atlassian.com/software/bitbucket/pricing"),
]

COORDINATOR_PROMPT = """You are a competitive research coordinator. Launch one research \\
subagent PER competitor IN PARALLEL using task(), then pass findings to the data-analyst \\
subagent for a chart and report. Do NOT use take_screenshot."""

RESEARCHER_PROMPT = """You are a web research specialist. Navigate to the company website, \\
find pricing and features, extract text. Do NOT use take_screenshot - use extract_text. \\
Return: Company name, Pricing tiers, Notable features, Limitations."""

ANALYST_PROMPT = """You are a data analyst. Use execute_code to build a pandas DataFrame + \\
matplotlib chart, save \'comparison_chart.png\', and write a concise markdown comparison report. \\
Pre-installed: pandas, matplotlib, numpy."""


@app.entrypoint
async def invoke(payload, context):
    """Each invocation spins up fresh Browser/Interpreter MicroVMs scoped to this session."""
    prompt = payload.get("prompt", "Compare pricing for GitHub, GitLab, and Bitbucket")
    log.info(f"Invoking Deep Agents coordinator with prompt: {prompt}")

    model = ChatBedrockConverse(
        model="us.anthropic.claude-sonnet-4-6",
        region_name=AWS_REGION,
        config=BotoConfig(read_timeout=300),
    )

    toolkits = []
    research_subagents = []
    for name, url in COMPETITORS:
        tk, tools = create_browser_toolkit(region=AWS_REGION)
        tk.session_manager.session_wait_timeout = 60.0
        toolkits.append(tk)
        research_subagents.append({
            "name": f"research-{name.lower()}",
            "description": f"Researches {name} by browsing {url}.",
            "system_prompt": RESEARCHER_PROMPT,
            "tools": tools,
        })

    ci_tk, ci_tools = await create_code_interpreter_toolkit(region=AWS_REGION)
    toolkits.append(ci_tk)

    analyst = {
        "name": "data-analyst",
        "description": "Analyzes competitor data, generates charts and reports.",
        "system_prompt": ANALYST_PROMPT,
        "tools": ci_tools,
    }

    agent = create_deep_agent(
        model=model,
        subagents=[*research_subagents, analyst],
        tools=[],
        system_prompt=COORDINATOR_PROMPT,
        name="competitive-research-coordinator",
        checkpointer=None,
        store=InMemoryStore(),
    )

    try:
        result = await agent.ainvoke(
            {"messages": [{"role": "user", "content": prompt}]},
            config={"configurable": {"thread_id": context.session_id}},
        )
        output = result["messages"][-1].content
        log.info("Coordinator finished")
        return {"result": output}
    finally:
        for tk in toolkits:
            try:
                await tk.cleanup()
            except Exception as e:
                log.warning(f"Toolkit cleanup warning: {e}")


if __name__ == "__main__":
    app.run()
'''

main_path = f"{PROJECT_NAME}/app/{PROJECT_NAME}/main.py"
with open(main_path, "w") as f:
    f.write(main_py)
print(f"✅ Wrote {main_path}")

## Step 4: Update dependencies and generate the lock file

The scaffolded `pyproject.toml` needs three changes:
1. Add `deepagents` as a dependency
2. Change `langchain-aws` to `langchain-aws[tools]` (brings in Browser + Interpreter toolkits)
3. Bump `requires-python` to `>=3.11` (required by `deepagents`)

We then run `uv lock` to generate the `uv.lock` file. **The Dockerfile requires this file** —
without it, the CodeBuild container build will fail with `"/uv.lock": not found`.

In [ ]:
from pathlib import Path

pyproject_path = Path(f"{PROJECT_NAME}/app/{PROJECT_NAME}/pyproject.toml")
contents = pyproject_path.read_text()

# Add the [tools] extra to langchain-aws and add deepagents
contents = contents.replace(
    '"langchain-aws >= 1.0.0",',
    '"langchain-aws[tools] >= 1.0.0",\n    "deepagents",',
)

# deepagents requires Python 3.11+; the template defaults to >=3.10
contents = contents.replace(
    'requires-python = ">=3.10"',
    'requires-python = ">=3.11"',
)

pyproject_path.write_text(contents)
print("✅ Updated pyproject.toml — added deepagents, langchain-aws[tools], requires-python>=3.11\n")
print(pyproject_path.read_text())

In [ ]:
# Generate the lock file — required by the Dockerfile's COPY step
!cd {PROJECT_NAME}/app/{PROJECT_NAME} && uv lock
print("\n✅ uv.lock generated")

## Step 5: Deploy

`agentcore deploy` packages the agent as an ARM64 container via AWS CodeBuild,
pushes to ECR, and provisions the Runtime via CDK/CloudFormation.

**First deploy takes 8-12 minutes** because CodeBuild builds the container from
scratch (pulling the base image, installing all Python dependencies, pushing to ECR).
Subsequent deploys are faster due to layer caching.

In [ ]:
# -y auto-confirms; -v shows resource-level CloudFormation events
%cd {PROJECT_NAME}
!agentcore deploy -y -v
%cd ..

## Step 6: Grant Browser and Interpreter permissions to the execution role

The CDK-generated execution role has permissions for Bedrock model invocation and
CloudWatch logging, but **not** for AgentCore Browser or Code Interpreter APIs.
Without these permissions, the agent will fail with `AccessDeniedException` when
trying to start browser sessions or interpreter sessions.

We attach an inline policy with `bedrock-agentcore:*` to the execution role.
For production, scope this down to specific actions like `StartBrowserSession`,
`StartCodeInterpreterSession`, `InvokeCodeInterpreter`, etc.

> **No redeploy needed.** IAM policy changes take effect immediately. We wait
> 30 seconds for propagation before invoking.

In [ ]:
import json
import subprocess
import time

# Auto-detect the execution role created by CDK
result = subprocess.run(
    ["aws", "iam", "list-roles",
     "--query", "Roles[?contains(RoleName, 'ApplicationAgentDeepResea')].RoleName",
     "--output", "json"],
    capture_output=True, text=True
)
roles = json.loads(result.stdout)

if roles:
    exec_role_name = roles[0]
    print(f"Found execution role: {exec_role_name}")
else:
    # Fallback: list candidates and ask
    print("Could not auto-detect the role. Candidates:")
    !aws iam list-roles --query "Roles[?contains(RoleName, 'DeepResearch')].RoleName" --output table
    exec_role_name = input("Paste the execution role name: ").strip()

# Attach bedrock-agentcore:* permissions
policy = json.dumps({
    "Version": "2012-10-17",
    "Statement": [{
        "Sid": "AgentCoreBrowserInterpreter",
        "Effect": "Allow",
        "Action": "bedrock-agentcore:*",
        "Resource": "*"
    }]
})

subprocess.run([
    "aws", "iam", "put-role-policy",
    "--role-name", exec_role_name,
    "--policy-name", "BrowserInterpreterAccess",
    "--policy-document", policy,
], check=True)

print(f"\n✅ Policy attached to {exec_role_name}")
print("⏳ Waiting 30 seconds for IAM propagation...")
time.sleep(30)
print("Ready to invoke.")

## Step 7: Invoke the deployed agent

The CLI invokes the Runtime endpoint. The `--stream` flag shows the response as
it's generated.

**Expected runtime: 8-10 minutes** on first invocation. This includes:
- Runtime container cold start (~30-60s)
- 3 parallel browser MicroVM sessions browsing competitor websites
- 1 interpreter MicroVM session generating the comparison chart

The benefit of Runtime is managed hosting, per-user session isolation, and a stable
endpoint — not latency reduction over local execution.

In [ ]:
%cd {PROJECT_NAME}
!agentcore invoke --stream "Compare pricing for GitHub, GitLab, and Bitbucket"
%cd ..

## Step 8: Observability

The CLI auto-enables CloudWatch Transaction Search during deploy. Use these commands
to tail logs and inspect traces from the deployed agent:

In [ ]:
%cd {PROJECT_NAME}

# Last hour of logs
!agentcore logs --since 1h

# Recent traces
!agentcore traces list --limit 5

%cd ..

## Step 9: Clean up deployed resources

`agentcore remove all` resets the project config. The follow-up `agentcore deploy`
detects the empty state and tears down the CloudFormation stack — removing the
Runtime, ECR repository, IAM roles, and log groups.

In [ ]:
%cd {PROJECT_NAME}
!agentcore remove all -y
!agentcore deploy -y
%cd ..

# The inline policy is cleaned up when CloudFormation deletes the role.
# If the role still exists (e.g., partial teardown), remove it explicitly:
try:
    subprocess.run([
        "aws", "iam", "delete-role-policy",
        "--role-name", exec_role_name,
        "--policy-name", "BrowserInterpreterAccess",
    ], capture_output=True)
except Exception:
    pass

print("\n✅ Deployed resources torn down. You can delete the local project folder if you're done.")

## Troubleshooting Part 2

| Problem | Cause | Fix |
|---|---|---|
| `Target "default" not found in aws-targets.json` | `aws-targets.json` not populated | Run the Step 2 cell to auto-populate account/region |
| `"/uv.lock": not found` during CodeBuild | `uv lock` was not run before deploy | Run the `uv lock` cell in Step 4 |
| `uv lock` fails with `requires-python` error | `deepagents` requires Python ≥3.11 | Ensure Step 4 updated `requires-python` to `>=3.11` |
| `AccessDeniedException: StartBrowserSession` | Execution role missing Browser permissions | Run Step 6 to attach `bedrock-agentcore:*` policy |
| `403 Forbidden: User is not authorized to access automation stream` | Execution role has narrow permissions | Use `bedrock-agentcore:*` as in Step 6 (covers WebSocket actions) |
| `500` error from invoke | Runtime error in agent code | Check logs: `agentcore logs --since 1h --level error` |
| Deploy fails with `ROLLBACK_COMPLETE` | Previous failed stack still exists | Delete it: `aws cloudformation delete-stack --stack-name AgentCore-DeepResearchAgent-default --region us-west-2`, wait 60s, redeploy |
| Invoke takes 8-10 minutes | Expected for first invocation | Includes cold start + 3 browser sessions + interpreter |

## Notes and next steps for Part 2

- **Session isolation:** each `agentcore invoke` gets a unique session with its own MicroVMs. To continue a conversation across invocations, pass `--session-id <id>` to `agentcore invoke`.
- **Updating the agent:** edit `app/DeepResearchAgent/main.py` or `pyproject.toml`, run `uv lock`, then `agentcore deploy` again. A new immutable version is created; the `DEFAULT` endpoint auto-updates.
- **Model swap:** edit the `ChatBedrockConverse` line in `main.py` (or use `ChatAnthropic` / `ChatOpenAI` / `ChatGoogleGenerativeAI`) and redeploy.
- **Scoping down permissions:** for production, replace `bedrock-agentcore:*` with specific actions (`StartBrowserSession`, `GetBrowserSession`, `StopBrowserSession`, `StartCodeInterpreterSession`, `InvokeCodeInterpreter`, `StopCodeInterpreterSession`, `SendBrowserCommand`).
- **Memory on Runtime:** to use AgentCore Memory in the deployed version, add it via the CLI: `agentcore add memory --name ResearchMemory --strategies SEMANTIC`, then `agentcore deploy`. The memory ID is injected as an environment variable for your agent code to read.
- **Invoke from your own app:** use `boto3.client('bedrock-agentcore').invoke_agent_runtime(...)` with the agent ARN from `agentcore status`.
- **Pricing:** while the Runtime is deployed you pay for microVM time only when it's active. Sessions auto-terminate after 15 minutes idle (configurable) and have an 8-hour max lifetime.